# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** OpenCode/reporter exited with status 1; see run log p-9f5bf7e781bc39de6716e20287fa11fa-20261009T105338Z.log.
直近のツールエラー（最終原因との一致はログで確認）:
IndexError: list index out of range
IndexError: list index out of range
Error: {"message":"CUDA error: CUBLAS_STATUS_INTERNAL_ERROR when calling `cublasGemmEx( handle, opa, opb, m, n, k, &falpha, a, CUDA_R_16BF, lda, b, CUDA_R_16BF, ldb, &fbeta, c, std::is_same_v<C_Dtype, float> ? CUDA_R_32F : CUDA_R_16BF, ldc,

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: Assessing automatic data processing algorithms for RGB-D cameras to predict fruit size and weight in apples

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-9f5bf7e781bc39de6716e20287fa11fa)


# Apple fruit size estimation and weight prediction from RGB-D images — AK_SW_BENCHMARKER reproduction

Reproduction of the benchmarking subsection of:

**Miranda, J.C., Arnó, J., Gené-Mola, J., Lordan, J., Asín, L., Gregorio, E. (2023).
"Assessing automatic data processing algorithms for RGB-D cameras to predict fruit size and weight in apples."
*Computers and Electronics in Agriculture* 214, 108302.** DOI: [10.1016/j.compag.2023.108302](https://doi.org/10.1016/j.compag.2023.108302)

Author tool: [AK_SW_BENCHMARKER](https://github.com/GRAP-UdL-AT/ak_sw_benchmarker) (MIT license), pinned commit `5ef6bd5b86ae107ff87b12a15c96512c66dbc196`.

**What this notebook does (partial demonstration):** it runs the authors' own benchmarking code on the
bundled 6-frame Azure Kinect sample dataset (`AK_Story_RGB_IR_DEPTH_hour_5f_mask`) with the authors'
ground-truth fruit measurements. Per fruit it estimates the major/minor geometric axes (bounding box,
enclosing circle, fitted circle, fitted ellipse or rotated rectangle on the reference mask), estimates
the camera–fruit depth, converts pixels to mm via thin-lens theory, predicts weight with the paper's
allometric models, and reports MAE/RMSE/MAPE/R² against the ground truth — including the paper's
**occluded vs. non-occluded** split. A calibration-sphere plausibility check and a small
sizing×depth×weight-model ranking are included.

**What this notebook does NOT do:** it is *not* the paper's full orchard experiment (hundreds of fruits,
train/test protocol). Sample-level metrics here are **not comparable** to the paper's dataset-level
figures (reported: R² > 0.942, RMSE < 16 g, MAPE 5.1–5.7%).

**Runtime: CPU only** — no GPU is used anywhere in this pipeline (OpenCV contours + regression
coefficients). Select `Runtime > Change runtime type > CPU` (the default is fine). Expected wall time:
about 5–10 minutes; downloads ≈ 55 MB, all onto the Colab VM.

*(Japanese)* このノートブックは、原著論文のベンチマーク部分を著者自身のツール
AK_SW_BENCHMARKER（コミット 5ef6bd5b86ae107ff87b12a15c96512c66dbc196）で再現します。同梱の6フレームサンプルと正解データを用い、
ROIサイズ推定・深度推定・薄レンズ理論によるmm換算・アロメトリック重量予測・誤差指標(MAE/RMSE/MAPE/R²)を
計算します。論文の全実験（多数の果実・学習/検証分割）は対象外で、サンプル規模の指標は論文の数値と直接比較できません。
GPUは不要で、CPUのみで約5〜10分で実行できます（ダウンロード約55MB）。

## 1. Environment check

The pinned author code needs opencv (contour fitting), scipy (MATLAB depth files), pandas, scikit-learn
(R²) and lxml (PASCAL VOC parsing). All are preinstalled in current Colab runtimes (verified: Python
3.13, cv2 5.0, scipy 1.16); this cell fails loudly if anything essential is missing.

*(Japanese)* 必要ライブラリの有無と版を確認します。不足があればここで明示的に停止します。

In [ ]:
import sys
print("python", sys.version.split()[0])
required = ["cv2", "scipy", "pandas", "sklearn", "lxml", "matplotlib", "numpy"]
versions = {}
for mod in required:
    m = __import__(mod)
    versions[mod] = getattr(m, "__version__", "n/a")
for k, v in versions.items():
    print(f"{k:12s} {v}")
missing = [k for k in required if k not in versions]
assert not missing, f"missing packages: {missing} (install with pip in an earlier cell if needed)"


## 2. Pinned author source (code only, no dataset bytes)

We sparse-checkout only the `src/` tree of GRAP-UdL-AT/ak_sw_benchmarker at the paper-matched commit
`5ef6bd5b86ae107ff87b12a15c96512c66dbc196` (2023-10-06, "Fix doc"), assert the checked-out SHA, and import the authors'
**non-GUI** entry points. The paper's workflow was driven from the desktop GUI; per the GUI source
(`src/gui_benchmarking/gui_tab_window.py`, "Analyse dataset" → `PredictionMetricsFramework`), the exact
same computation is available as a reusable library, which is what we call below. The GUI is not
reproduced — only its scientific operations.

*(Japanese)* 著者リポジトリの `src/` のみをピン留したコミットで取得し、GUI本体ではなくGUIが呼び出す
のと同じ非GUI計算モジュールを利用します。

In [ ]:
import subprocess, os, sys, pathlib
SRC_DIR = "/content/ak_sw_benchmarker"
COMMIT = "5ef6bd5b86ae107ff87b12a15c96512c66dbc196"
if not pathlib.Path(SRC_DIR).exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                    "https://github.com/GRAP-UdL-AT/ak_sw_benchmarker.git", SRC_DIR], check=True)
head = subprocess.run(["git", "-C", SRC_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
if head != COMMIT:
    subprocess.run(["git", "-C", SRC_DIR, "fetch", "--filter=blob:none", "origin", COMMIT], check=True)
    subprocess.run(["git", "-C", SRC_DIR, "checkout", "--detach", COMMIT], check=True)
subprocess.run(["git", "-C", SRC_DIR, "sparse-checkout", "set", "src"], check=True)
subprocess.run(["git", "-C", SRC_DIR, "checkout", COMMIT], check=True)
head = subprocess.run(["git", "-C", SRC_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
assert head == COMMIT, f"checkout mismatch: {head}"
print("author source checked out at", head)
sys.path.insert(0, os.path.join(SRC_DIR, "src"))
from dataset_management.dataset_config import DatasetConfig
from dataset_management.dataset_manager import DatasetManager
from dataset_management.pascal_voc_parser import PascalVocParser
from data_features_processor.data_features_config import DataFeatureConfig
from data_features_processor.data_features_config import SizeEstimationSelectorPx
from depth_estimation_s.depth_estimation_methods_selector import DepthSelector
from size_estimation_s.image_processing import ImageProcessing
from size_estimation_s.size_estimation_methods_selector import SizeEstimationSelectorPx
from size_estimation_s.roi_selector import ROISelector
from weight_prediction_s.weight_prediction_methods import WeightPredictionModels
from weight_prediction_s.weight_prediction_methods_selector import WeightPredictionModelSelector
from camera_management_s.camera_parameters import AzureKinect
from reports_management.prediction_metrics_framework import PredictionMetricsFramework
from reports_management.prediction_metrics_framework_config import PredictionsMetricsFrameworkConfig
from reports_management.comparative_measures_reports import ComparativeMeasuresReport
from reports_management.comparative_measures_reports_selector import ComparativeMeasuresReportSelector
import pandas as pd
print("author modules import OK")

### Compatibility note — `scipy.stats.mode` return shape

The pinned author code indexes the mode result as `stats.mode(values, axis=None)[0][0]`
(`src/depth_estimation_s/depth_estimation_methods.py:51,91,127`), which matches the tuple-of-arrays
return of scipy < 1.9. Current Colab ships scipy 1.16, where `mode()` returns a scalar `mode` field and
that indexing raises `IndexError`. We install a minimal wrapper that restores exactly the old return
shape (array of mode values, array of counts) so the author's unmodified code runs. This shim changes
no numerical behavior — it only adapts the return-value container.

*(Japanese)* 著者コードはscipy 1.9未満の`mode()`戻り値形状を前提としているため、
同じ要素を返す最小限の互換ラッパーをここで用意します（数値動作は変わりません）。

In [ ]:
import numpy as np
import scipy.stats
_orig_scipy_mode = scipy.stats.mode
def _scipy_mode_compat(a, axis=None, **kwargs):
    res = _orig_scipy_mode(a, axis=axis, **kwargs)
    return np.array([np.atleast_1d(res.mode), np.atleast_1d(res.count)], dtype=object)
scipy.stats.mode = _scipy_mode_compat
import scipy
print("compat shim installed; scipy", scipy.__version__, "- author code indexes mode(...)[0][0] (scipy<1.9 shape)")

## 3. Sample dataset and ground truth (downloaded onto the Colab VM)

From the pinned commit we fetch, per frame: the RGB image `*_C.png`, the registered depth map
`*_D.mat` (MATLAB v5, internal key `transformed_depth`), the manually segmented binary mask `masks/*_C.png`,
the PASCAL VOC bounding-box annotations `square_annotations1/*_C.xml` and the per-fruit annotation CSVs.
Ground truth comes from the authors' `data/csv_datasets/dataset_with_spheres/`:
`..._STATIC_v2.csv` (all 28 fruits incl. two calibration spheres), and its `..._OC_v2.csv` (occluded, 19)
/ `..._VI_v2.csv` (non-occluded, 11) subsets — the paper's occlusion comparison. ≈50 MB total; every
byte stays under `/content` on this VM.

*(Japanese)* サンプル6フレーム（RGB/深度/マスク/アノテーション）と正解CSV一式をピン留したraw URLから
このVM内にのみダウンロードします（合計約50MB）。

In [ ]:
import urllib.request
RAW = "https://raw.githubusercontent.com/GRAP-UdL-AT/ak_sw_benchmarker/5ef6bd5b86ae107ff87b12a15c96512c66dbc196"
BASE = "data/AK_Story_RGB_IR_DEPTH_hour_5f_mask/preprocessed_data"
FRAMES = ['20210927_114012_k_r2_e_000_150_138_2_0_C', '20210927_115932_k_r2_e_000_150_138_2_0_C', '20210927_121804_k_r2_e_000_150_138_2_0_C', '20210927_123505_k_r2_e_000_150_138_2_0_C', '20210927_125309_k_r2_e_000_150_138_2_0_C', '20210927_192424_k_r2_e_000_150_138_2_0_C']
GT_FILES = {'with_spheres': 'fruit_measures_TREES_01-03_EAST_SIDE_STATIC_v2.csv', 'occluded': 'fruit_measures_TREES_01-03_EAST_SIDE_STATIC_OC_v2.csv', 'visible': 'fruit_measures_TREES_01-03_EAST_SIDE_STATIC_VI_v2.csv'}
DS = pathlib.Path("/content/ak_sw_sample/AK_Story_RGB_IR_DEPTH_hour_5f_mask")
PRE = DS / "preprocessed_data"
for sub in ["images", "masks", "square_annotations1", "annotations"]:
    (PRE / sub).mkdir(parents=True, exist_ok=True)
GT_DIR = pathlib.Path("/content/ak_sw_gt"); GT_DIR.mkdir(exist_ok=True)

def fetch(rel, dest):
    dest = pathlib.Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest.stat().st_size
    tmp = dest.with_suffix(dest.suffix + ".part")
    urllib.request.urlretrieve(f"{RAW}/{rel}", tmp)
    tmp.rename(dest)
    return dest.stat().st_size

total = 0
for F in FRAMES:
    total += fetch(f"{BASE}/images/{F}.png", PRE / "images" / f"{F}.png")
    total += fetch(f"{BASE}/images/{F[:-1]}D.mat", PRE / "images" / f"{F[:-1]}D.mat")
    total += fetch(f"{BASE}/masks/{F}.png", PRE / "masks" / f"{F}.png")
    total += fetch(f"{BASE}/square_annotations1/{F}.xml", PRE / "square_annotations1" / f"{F}.xml")
    total += fetch(f"{BASE}/annotations/{F}.csv", PRE / "annotations" / f"{F}.csv")
for tag, name in GT_FILES.items():
    total += fetch(f"data/csv_datasets/dataset_with_spheres/{name}", GT_DIR / f"{tag}.csv")
print(f"downloaded {total/1e6:.1f} MB")

pngs = sorted((PRE / "images").glob("*_C.png")); mats = sorted((PRE / "images").glob("*_D.mat"))
masks = sorted((PRE / "masks").glob("*.png")); xmls = sorted((PRE / "square_annotations1").glob("*.xml"))
assert len(pngs) == 6 and len(mats) == 6 and len(masks) == 6 and len(xmls) == 6, (len(pngs), len(mats), len(masks), len(xmls))
for p in pngs + masks:
    assert p.read_bytes()[:8] == b"\x89PNG\r\n\x1a\n", f"not a PNG: {p}"
for m in mats:
    assert m.read_bytes()[:8] == b"MATLAB 5", f"not a MATLAB v5 file: {m}"
for g in GT_DIR.glob("*.csv"):
    assert b"fruit_id" in g.read_bytes(), f"ground truth missing fruit_id: {g}"
print("6 frames verified: PNG/MATLAB-5 signatures OK, ground-truth CSVs contain fruit_id")

## 4. Input preview — sample frame with reference annotations

Before any measurement we look at the actual data. The overlay below shows the raw RGB frame with the
authors' reference binary mask (red, semi-transparent) and the PASCAL VOC bounding boxes with fruit
ids (yellow). These annotations are the **reference labels** used by the benchmarker's ROI stage — they
are not our predictions. Two `BALL_*` objects are calibration spheres of known diameter (60 mm and
120 mm) that we use later as a plausibility anchor.

*(Japanese)* 実データのプレビュー：RGB画像に著者の参照マスク（赤半透明）と
PASCAL VOCバウンディングボックス（黄、果実ID付き）を重ねて表示します。BALL_*は直径既知の校正球です。

In [ ]:
import numpy as np
import cv2
import scipy.io as sio
import matplotlib.pyplot as plt
F0 = FRAMES[0]
rgb = cv2.imread(str(PRE / "images" / f"{F0}.png"))[:, :, ::-1]
mask = cv2.imread(str(PRE / "masks" / f"{F0}.png"), cv2.IMREAD_GRAYSCALE)
boxes, labels = PascalVocParser.readXMLFromFile(str(PRE / "square_annotations1" / f"{F0}.xml"))
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(11, 6.5))
ax.imshow(rgb)
ax.imshow(np.ma.masked_where(mask == 0, mask), cmap="autumn", alpha=0.35)
for (xmin, ymin, xmax, ymax), lab in zip(boxes, labels):
    ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, fill=False, edgecolor="yellow", lw=1.0))
    ax.text(xmin, ymin - 3, str(lab), color="yellow", fontsize=6,
            bbox=dict(facecolor="black", alpha=0.4, pad=0.5, lw=0))
ax.set_title(f"Reference input: frame {F0}\nRGB + author reference mask (red) + PASCAL VOC boxes (yellow)")
ax.axis("off")
plt.tight_layout(); plt.show()
print(f"frame {F0}: RGB {rgb.shape}, depth range "
      f"{sio.loadmat(str(PRE / 'images' / (F0[:-1] + 'D.mat')))['transformed_depth'].min()}-"
      f"{sio.loadmat(str(PRE / 'images' / (F0[:-1] + 'D.mat')))['transformed_depth'].max()} mm, "
      f"{len(boxes)} labelled objects")

## 5. Ground-truth fruit measurements

The ground truth files map each `fruit_id` to laboratory-measured values (`lab.*`): camera–fruit depth,
caliper-measured caliber/height (`o_caliber_mm`/`o_height_mm`) and geometric axes
(`axis_01_mm` = major, `axis_02_mm` = minor) plus `weight_gr`. We load them exactly as the authors' GUI
does (`gui_tab_window.py:682-690`: `;`-separated, `fruit_id` as string, `lab.*` as float).

*(Japanese)* 正解CSVを実データとして読み込み、列と先頭行を表示します（`;`区切り、GUIと同じ型変換）。

In [ ]:
gt = {}
for tag in GT_FILES:
    df = pd.read_csv(GT_DIR / f"{tag}.csv", sep=";", dtype=str)
    gt[tag] = df
    print(f"{tag:13s} {df.shape[0]:3d} fruits, columns: {list(df.columns)}")
LAB_COLS = ["lab.depth_mm", "lab.o_caliber_mm", "lab.o_height_mm",
            "lab.axis_01_mm", "lab.axis_02_mm", "lab.weight_gr"]
def load_measures(tag):
    df = gt[tag].copy()
    df["fruit_id"] = df["fruit_id"].astype(str)
    for c in LAB_COLS:
        df[c] = df[c].astype(float)
    return df
measures_df = load_measures("with_spheres")
print()
print(measures_df[["fruit_id", "lab.axis_01_mm", "lab.axis_02_mm", "lab.weight_gr"]].head(8).to_string(index=False))

## 6. Headline benchmark run — the authors' pipeline on the sample

This is the authors' own benchmarking loop (`PredictionMetricsFramework.comparative_metrics_dataset_mask`,
the exact function their GUI calls): for every frame it loads RGB + `transformed_depth` + mask, erode→dilate
cleans the mask (`ImageProcessing.im_method_1`), and for each PASCAL VOC object computes
pixel axes → ROI depth (`AVG` over masked pixels) → thin-lens mm (Azure Kinect RGB focal length 1040 px,
`AzureKinect().rgb_sensor`) → allometric weight `Y = β₀ + β₁·D₁ + β₂·D₂` (`D1D2_LM_MET_03`,
coefficients from the pinned source: β₀=-161.64, β₁=2.48, β₂=2.22) → merge with ground truth on
`fruit_id` → MAE/RMSE/MAPE/R². Expected output: printed error metrics for axes and weight over the
6-frame sample.

*(Japanese)* 著者自身のベンチマーク関数を、マスクROI・BBサイズ推定・AVG深度・
両軸線形モデル(D1D2_LM_MET_03)で6フレーム全体に実行し、誤差指標を表示します。

In [ ]:
import scipy.io as sio  # noqa: F401  (author loop uses sio internally)
DATASET_PARENT = "/content/ak_sw_sample"
DATASET_NAME = "AK_Story_RGB_IR_DEPTH_hour_5f_mask"
OUT_DIR = pathlib.Path("/content/ak_sw_out"); OUT_DIR.mkdir(exist_ok=True)
dataset_manager_config = DatasetConfig(DATASET_PARENT, DATASET_NAME)
data_features_options = DataFeatureConfig(
    camera_conf=AzureKinect().rgb_sensor,
    roi_selector=ROISelector.MASK,
    size_estimation_selector=SizeEstimationSelectorPx.BB,
    depth_selector=DepthSelector.AVG,
    weight_selector=WeightPredictionModelSelector.D1D2_LM_MET_03)
benchmarker_config = PredictionsMetricsFrameworkConfig(
    dataset_manager_config, str(OUT_DIR / "headline_comparative.csv"),
    data_features_options, WeightPredictionModelSelector.D1D2_LM_MET_03,
    ComparativeMeasuresReportSelector.WEIGHT)
headline_fw = PredictionMetricsFramework(benchmarker_config)
headline_fw.comparative_metrics_dataset_mask(measures_df)
headline_report = headline_fw.a_report
def metrics_row(name, a1, a2, w):
    return {"combination": name,
            "axis01_RMSE_mm": round(a1.RMSE, 2), "axis01_MAPE_%": round(100 * a1.MAPE, 2),
            "axis02_RMSE_mm": round(a2.RMSE, 2), "axis02_MAPE_%": round(100 * a2.MAPE, 2),
            "weight_RMSE_g": round(w.RMSE, 2), "weight_MAPE_%": round(100 * w.MAPE, 2),
            "weight_R2": round(w.R2, 4)}
headline_rows = [metrics_row("MASK+BB, AVG depth, D1D2_LM_MET_03 (all 28 fruits)",
                             headline_report.get_axis_01_metrics(),
                             headline_report.get_axis_02_metrics(),
                             headline_report.get_weight_metrics())]
pd.DataFrame(headline_rows)

## 7. Per-fruit results and calibration-sphere plausibility check

Per-fruit means across the 6 frames (each fruit appears once per frame). Two checks follow: the
`BALL_060`/`BALL_120` calibration spheres of known 60 mm/120 mm diameter should be measured near their
nominal size, and predicted weight should track laboratory weight. The scatter plot is generated by
this notebook from the actual sample run (it is not a figure from the paper).

*(Japanese)* 果実ごとの6フレーム平均を表にし、既知直径の校正球(BALL_060/BALL_120)の妥当性確認と、
予測重量-vs-実測重量の散分図（このノートブックで生成した追加図）を示します。

In [ ]:
comp = headline_report.get_comparative_df()
per_fruit = (comp.groupby("fruit_id")[["lab.axis_01_mm", "pred.axis_01_mm", "lab.axis_02_mm",
                                       "pred.axis_02_mm", "lab.weight_gr", "pred.weight_gr"]]
             .mean().round(2))
display(per_fruit)
spheres = per_fruit.loc[[i for i in ["BALL_060", "BALL_120"] if i in per_fruit.index]]
print("Calibration spheres (nominal 60 mm / 120 mm):")
print(spheres[["lab.axis_01_mm", "pred.axis_01_mm"]].to_string())
fig, ax = plt.subplots(figsize=(6.5, 6))
is_sphere = per_fruit.index.str.startswith("BALL")
ax.scatter(per_fruit.loc[~is_sphere, "lab.weight_gr"], per_fruit.loc[~is_sphere, "pred.weight_gr"],
           s=45, label="apples (26)")
if is_sphere.any():
    ax.scatter(per_fruit.loc[is_sphere, "lab.weight_gr"], per_fruit.loc[is_sphere, "pred.weight_gr"],
               marker="s", s=70, label="calibration spheres (2)")
lims = [per_fruit["lab.weight_gr"].min() - 20, per_fruit["lab.weight_gr"].max() + 20]
ax.plot(lims, lims, "k--", lw=1, label="1:1")
ax.set_xlabel("laboratory weight (g)"); ax.set_ylabel("predicted weight (g)")
ax.set_title("Predicted vs laboratory weight per fruit\n(MASK+BB, AVG depth, D1D2_LM_MET_03; notebook-generated)")
ax.legend(); plt.tight_layout(); plt.show()

## 8. Occluded vs. non-occluded fruits

The paper's central comparison. The authors provide separate ground-truth subsets: occluded fruits
(`..._OC_v2.csv`, 19 fruits) and non-occluded ones (`..._VI_v2.csv`, 11 fruits). We rerun the same
pipeline against each subset (the pipeline is identical; only the merged ground truth changes) and
report the resulting error metrics. Expected pattern from the paper: occluded fruits yield larger
errors than non-occluded ones, but both remain usable.

*(Japanese)* 論文の中心比較：遮蔽あり(OC)となし(VI)の正解サブセットで同じパイプラインを再実行し
誤差指標を比較します。

In [ ]:
split_rows = []
for tag, label in [("with_spheres", "all fruits (26 apples + 2 spheres)"),
                   ("visible", "non-occluded only (VI)"),
                   ("occluded", "occluded only (OC)")]:
    md_ = load_measures(tag)
    cfg = PredictionsMetricsFrameworkConfig(
        DatasetConfig(DATASET_PARENT, DATASET_NAME), str(OUT_DIR / f"split_{tag}.csv"),
        DataFeatureConfig(camera_conf=AzureKinect().rgb_sensor, roi_selector=ROISelector.MASK,
                          size_estimation_selector=SizeEstimationSelectorPx.BB,
                          depth_selector=DepthSelector.AVG,
                          weight_selector=WeightPredictionModelSelector.D1D2_LM_MET_03),
        WeightPredictionModelSelector.D1D2_LM_MET_03, ComparativeMeasuresReportSelector.WEIGHT)
    fw = PredictionMetricsFramework(cfg)
    fw.comparative_metrics_dataset_mask(md_)
    rep = fw.a_report
    split_rows.append(metrics_row(f"{label} — n={len(md_)}", rep.get_axis_01_metrics(),
                                  rep.get_axis_02_metrics(), rep.get_weight_metrics()))
pd.DataFrame(split_rows)

## 9. Sizing × depth × weight-model ranking

The GUI's "Run tests in dataset" benchmarks all sizing/allometric combinations; we reproduce a bounded
version. For each (sizing, depth) pair the authors' framework runs once over the 6 frames; the weight
column is then recomputed for each allometric model with the authors' own `WeightPredictionModels.predict_weight`
and scored with their `ComparativeMeasuresReport`. Sizing methods: `BB` bounding box, `CE` enclosing
circle, `CF` fitted circle, `EF` fitted ellipse, `RR` rotated rectangle. Weight models include the
abstract's best "linear model with both axes" (`D1D2_LM_MET_03` / `CH_LM_MET_03`). Expected output: a
ranking table sorted by weight RMSE; values are sample-level, not paper-level.

*(Japanese)* GUIの「Run tests in dataset」に相当する小規模ランキング：サイズ推定法×深度オプションの
組合せで著者フレームワークを実行し、重量モデルは著者の`predict_weight`で再計算してRMSE順に並べます。

In [ ]:
SIZE_DEPTH_COMBOS = ([(s, DepthSelector.AVG) for s in
                      [SizeEstimationSelectorPx.BB, SizeEstimationSelectorPx.CE,
                       SizeEstimationSelectorPx.CF, SizeEstimationSelectorPx.EF,
                       SizeEstimationSelectorPx.RR]] +
                     [(SizeEstimationSelectorPx.BB, DepthSelector.MOD),
                      (SizeEstimationSelectorPx.BB, DepthSelector.MIN)])
WEIGHT_MODELS = [WeightPredictionModelSelector.D1D2_LM_MET_03, WeightPredictionModelSelector.CH_LM_MET_03,
                 WeightPredictionModelSelector.D1D2_LM_MET_01, WeightPredictionModelSelector.D1D2_NLM_MET_02]
rank_rows = []
for size_sel, depth_sel in SIZE_DEPTH_COMBOS:
    cfg = PredictionsMetricsFrameworkConfig(
        DatasetConfig(DATASET_PARENT, DATASET_NAME), str(OUT_DIR / "sweep_tmp.csv"),
        DataFeatureConfig(camera_conf=AzureKinect().rgb_sensor, roi_selector=ROISelector.MASK,
                          size_estimation_selector=size_sel, depth_selector=depth_sel,
                          weight_selector=WeightPredictionModelSelector.D1D2_LM_MET_03),
        WeightPredictionModelSelector.D1D2_LM_MET_03, ComparativeMeasuresReportSelector.WEIGHT)
    fw = PredictionMetricsFramework(cfg)
    fw.comparative_metrics_dataset_mask(measures_df)
    base_df = fw.a_report.get_comparative_df()
    a1m, a2m = fw.a_report.get_axis_01_metrics(), fw.a_report.get_axis_02_metrics()
    for wsel in WEIGHT_MODELS:
        df_w = base_df.copy()
        df_w["pred.weight_gr"] = WeightPredictionModels().predict_weight(
            df_w["pred.axis_01_mm"], df_w["pred.axis_02_mm"], wsel)
        rep = ComparativeMeasuresReport(df_w, ComparativeMeasuresReportSelector.WEIGHT)
        rank_rows.append(metrics_row(f"{size_sel.name}+{depth_sel.name}, {wsel.name}",
                                     a1m, a2m, rep.get_weight_metrics()))
ranking = pd.DataFrame(rank_rows).sort_values("weight_RMSE_g").reset_index(drop=True)
ranking

## 10. Sizing-method comparison (AVG depth, both-axes linear model)

A compact view of the ranking above: weight RMSE per sizing method at AVG depth with the paper's
best linear both-axes model. This bar chart is notebook-generated from the actual run (not an author
figure). Expected result: the five shape settings differ by a few grams of RMSE on this sample.

*(Japanese)* AVG深度・両軸線形モデルにおけるサイズ推定法別の重量RMSEを棒グラフで比較します
（このノートブックで生成した追加図）。

In [ ]:
sel = ranking[ranking["combination"].str.contains("AVG, D1D2_LM_MET_03")].copy()
sel["sizing"] = sel["combination"].str.split("+").str[0]
sel = sel.sort_values("weight_RMSE_g")
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(sel["sizing"], sel["weight_RMSE_g"], color="#4C72B0")
for x, (rmse, mape) in enumerate(zip(sel["weight_RMSE_g"], sel["weight_MAPE_%"])):
    ax.text(x, rmse, f"{rmse:.1f} g\n{mape:.1f}%", ha="center", va="bottom", fontsize=9)
ax.set_xlabel("sizing method (reference-mask based, AVG depth, D1D2_LM_MET_03)")
ax.set_ylabel("weight RMSE (g)")
ax.set_title("Weight RMSE by sizing method on the 6-frame sample (notebook-generated)")
plt.tight_layout(); plt.show()

## 11. Export results

We export the full predicted-vs-laboratory comparative datasheet of the headline run and the ranking
table as CSV files on the VM (`/content/ak_sw_out/`), in the same spirit as the tool's `export_data_csv`.

*(Japanese)* 代表実行の比較データシートとランキング表を `/content/ak_sw_out/` にCSV保存します。

In [ ]:
headline_report.export_data_csv(str(OUT_DIR / "headline_comparative.csv"))
ranking.to_csv(OUT_DIR / "ranking.csv", index=False, float_format="%.3f")
for f in sorted(OUT_DIR.iterdir()):
    print(f.name, f.stat().st_size, "bytes")

## 12. Interpretation, scope and limitations

**What was executed:** the authors' own benchmarking code (pinned commit) on their bundled 6-frame
Azure Kinect sample with their ground truth — ROI sizing (5 shape settings), depth estimation
(AVG/MOD/MIN), thin-lens mm conversion, 4+ allometric weight models, and MAE/RMSE/MAPE/R² against
laboratory measurements, including the occluded vs. non-occluded split. The calibration spheres
anchor the mm-scale: `BALL_060`/`BALL_120` measurements printed above should sit near 60/120 mm.

**Scientific scope — partial demonstration.** This is *not* the paper's full experiment (its orchard
dataset, train/test protocol and reported dataset-level accuracy: R² > 0.942, RMSE < 16 g, MAPE
5.1–5.7% for non-occluded apples). Sample-level metrics in this notebook must not be compared to those
figures; with only 6 frames and ≤28 fruits they are plausibility evidence, not verification of the
published results.

**Provenance:** all measurements come from GRAP-UdL-AT/ak_sw_benchmarker functions executed unmodified;
the only notebook-added logic is orchestration (loops, ranking, plots, CSV export). Reference: paper
[DOI 10.1016/j.compag.2023.108302](https://doi.org/10.1016/j.compag.2023.108302); tool SoftwareX paper
[DOI 10.1016/j.softx.2023.101548](https://doi.org/10.1016/j.softx.2023.101548); MIT license.

*(Japanese)* 実行内容は著者コードそのものによるサンプル規模の再現です。校正球によるmmスケールの
妥当性確認を含みます。6フレーム・28果実規模の指標は論文のデータセット規模の成果（R²>0.942、RMSE<16g、
MAPE 5.1–5.7%）の検証にはならず、あくまで妥当性の目安です。すべての測定は著者コードを無改変で実行した
結果であり、ノートブックが追加したのはループ・ランキング・図・CSV出力の操作のみです。